In [ ]:
# Program 6: K-Means Clustering on Sales Data
# Dataset: https://www.kaggle.com/datasets/kyanyoga/sample-sales-data (sales_data_sample.csv)

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
# Load dataset
df = pd.read_csv('sales_data_sample.csv', encoding='latin1')
features = ['QUANTITYORDERED', 'PRICEEACH', 'SALES', 'MSRP']
df_cluster = df[features].dropna().copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cluster)
print(f"Scaled shape: {X_scaled.shape}")
df_cluster.head()

In [ ]:
# Elbow Method & Silhouette Analysis
inertias = []
silhouette_scores = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, km.labels_)
    silhouette_scores.append(sil)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(k_range, inertias, 'bo-', linewidth=2)
ax1.set_title('Elbow Method')
ax1.set_xlabel('K')
ax1.set_ylabel('Inertia (WCSS)')
ax1.grid(True, alpha=0.3)

ax2.plot(k_range, silhouette_scores, 'rs-', linewidth=2)
ax2.set_title('Silhouette Score vs K')
ax2.set_xlabel('K')
ax2.set_ylabel('Silhouette Score')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Final Clustering
optimal_k = 3
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
clusters = kmeans_final.fit_predict(X_scaled)
df_cluster['Cluster'] = clusters

print(f"Cluster counts (K={optimal_k}):\n", df_cluster['Cluster'].value_counts())

centroids = scaler.inverse_transform(kmeans_final.cluster_centers_)
centroids_df = pd.DataFrame(centroids, columns=features, index=[f"Cluster {i}" for i in range(optimal_k)])
print("\nCluster Centroids:\n", centroids_df.round(2))

plt.figure(figsize=(9, 6))
scatter = plt.scatter(df_cluster['SALES'], df_cluster['QUANTITYORDERED'],
                      c=df_cluster['Cluster'], cmap='viridis', alpha=0.6, edgecolors='k', linewidth=0.5)
plt.colorbar(scatter, label='Cluster ID')
plt.title(f'K-Means Clustering (K={optimal_k})')
plt.xlabel('Sales')
plt.ylabel('Quantity Ordered')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()